In [1]:
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Setup & Load Model
nltk.download('punkt')
model_path = r"B:\stacks\Machine_learning_projects\Video_Summarization_Translation\src\deberta-ytseg-finetuned\checkpoint-15000"  # Path to your saved model
model_id = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()
if torch.cuda.is_available():
    model.to("cuda")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\MauryaP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
def segment_paragraph(text, threshold=0.5):
    sentences = nltk.sent_tokenize(text)
    if len(sentences) < 2:
        return [text]

    # 1. Prepare all triplets for batch processing
    encoded_inputs = []
    sep = tokenizer.sep_token
    
    for i in range(len(sentences)):
        target = sentences[i]
        prev_s = sentences[i-1] if i > 0 else "[START]"
        next_s = sentences[i+1] if i < len(sentences)-1 else "[END]"
        
        # We store the raw pairs to batch-tokenize them later
        encoded_inputs.append((target, f"{prev_s} {sep} {next_s}"))

    # 2. Batch Tokenization
    inputs = tokenizer(
        [pair[0] for pair in encoded_inputs],
        [pair[1] for pair in encoded_inputs],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    # 3. Inference
    if torch.cuda.is_available():
        model.to("cuda")
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        # We use softmax to get probabilities for the '1' (boundary) class
        probs = F.softmax(outputs.logits, dim=-1)
        # probs[:, 1] is the probability of a topic boundary
        predictions = (probs[:, 1] > threshold).int().tolist()

    # 4. Reconstruct with logic
    # Note: Usually, the very first sentence is always the start of Topic 1, 
    # so we ignore a '1' prediction on the 0th index.
    segments = []
    current_segment = [sentences[0]]

    for i in range(1, len(sentences)):
        if predictions[i] == 1:
            segments.append(" ".join(current_segment))
            current_segment = [sentences[i]]
        else:
            current_segment.append(sentences[i])
    
    segments.append(" ".join(current_segment))
    return segments, probs[:, 1].tolist()

In [ ]:
# --- Example Usage ---
with open(r"..\texts\K-Means_Clustering_Algorithm_From_Scratch_In_Python__ML_Algorithms_From_Scratch.txt") as file:
    paragraph = file.read()
# paragraph = """
# Artificial Intelligence has changed how we write code. Many developers use Copilot daily. 
# However, there are risks regarding data privacy. Companies are now creating internal LLMs. 
# Moving on to another subject, space exploration is reaching new heights. 
# SpaceX recently launched another Starship. NASA is planning the next moon mission.
# """

topics, scores = segment_paragraph(paragraph, threshold=0.008)

# for i, (sent, score) in enumerate(zip(nltk.sent_tokenize(paragraph), scores)):
#     print(f"Score: {score:.4f} | {sent[:50]}...")

print(f"\nTotal Segments found: {len(topics)}")

for idx, topic in enumerate(topics):
    print(f"--- TOPIC {idx+1} ---")
    print(topic, "\n")


Total Segments found: 9
--- TOPIC 1 ---
 Hello guys, my name is Nitish, welcome to my YouTube channel. So we were reading the K-Means clustering algorithm. In the last two videos, we applied the geometric intuition of this algorithm and a practical dataset. In this video, I have planned to code this algorithm from scratch from end to end using Python. Because I always feel that if you want to do good in machine learning and you want to understand the algorithm very well, then the best way is to try to code it from scratch. If you can code it from scratch, then you have got the algorithm. So K-Means is very interesting and it will be fun to code it. So let's start the video. So yeah guys, let's start with the coding part. I have generated a dataset, we will work on it. If I run the code right now, then at this point I have a dataset with two clusters like this. And we will write the code of K-Means from scratch and we will identify these two clusters. It is a very simple code. I took t